In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("anshulm257/rice-disease-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'rice-disease-dataset' dataset.
Path to dataset files: /kaggle/input/rice-disease-dataset


In [ ]:
import os

base_path = "/kaggle/input/rice-disease-dataset"
print(os.listdir(base_path))

['Rice_Leaf_AUG']


In [ ]:
import os

folder = base_path + "/Rice_Leaf_AUG"
print(os.listdir(folder)[:20])


['Leaf scald', 'Sheath Blight', 'Healthy Rice Leaf', 'Leaf Blast', 'Brown Spot', 'Bacterial Leaf Blight']


In [ ]:
import os
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch

base_path = "/kaggle/input/rice-disease-dataset"
print(os.listdir(base_path))
data_dir = base_path + "/Rice_Leaf_AUG"

# Image Augmentation + Preprocessing
transform = {
    "train": transforms.Compose([
        transforms.Resize((224,224)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(20),
        transforms.ToTensor()
    ]),
    "test": transforms.Compose([
        transforms.Resize((224,224)),
        transforms.ToTensor()
    ]),
}

# Auto split train/test
dataset = datasets.ImageFolder(data_dir, transform["train"])

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print("Classes:", dataset.classes)
print("Train samples:", len(train_dataset))
print("Test samples:", len(test_dataset))

['Rice_Leaf_AUG']
Classes: ['Bacterial Leaf Blight', 'Brown Spot', 'Healthy Rice Leaf', 'Leaf Blast', 'Leaf scald', 'Sheath Blight']
Train samples: 3063
Test samples: 766


In [ ]:
import torch
import torch.nn as nn
from torchvision import models

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

model = models.resnet18(weights="IMAGENET1K_V1")
model.fc = nn.Linear(model.fc.in_features, len(dataset.classes))  # number of classes
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)


Device: cuda
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 198MB/s]


In [ ]:
from tqdm import tqdm

epochs = 10

for epoch in range(epochs):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    print(f"\n===== Epoch {epoch+1}/{epochs} =====")

    for images, labels in tqdm(train_loader, desc=f"Training Epoch {epoch+1}"):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        # accuracy
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_loss = total_loss / len(train_loader)
    epoch_acc = 100 * correct / total

    print(f"Epoch {epoch+1} → Loss: {epoch_loss:.4f} | Accuracy: {epoch_acc:.2f}%")



===== Epoch 1/10 =====


Training Epoch 1: 100%|██████████| 96/96 [01:09<00:00,  1.38it/s]


Epoch 1 → Loss: 0.8038 | Accuracy: 71.86%

===== Epoch 2/10 =====


Training Epoch 2: 100%|██████████| 96/96 [00:45<00:00,  2.13it/s]


Epoch 2 → Loss: 0.2793 | Accuracy: 91.48%

===== Epoch 3/10 =====


...

Epoch 8 → Loss: 0.0448 | Accuracy: 98.89%

===== Epoch 9/10 =====


Training Epoch 9: 100%|██████████| 96/96 [00:44<00:00,  2.14it/s]


Epoch 9 → Loss: 0.0440 | Accuracy: 98.76%

===== Epoch 10/10 =====


Training Epoch 10: 100%|██████████| 96/96 [00:44<00:00,  2.16it/s]

Epoch 10 → Loss: 0.0328 | Accuracy: 99.12%


In [ ]:
model.eval()
correct, total = 0, 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Test Accuracy: {100 * correct / total:.2f}%")


Test Accuracy: 96.74%


In [ ]:
import os

print(os.listdir(data_dir))


['Leaf scald', 'Sheath Blight', 'Healthy Rice Leaf', 'Leaf Blast', 'Brown Spot', 'Bacterial Leaf Blight']


In [ ]:
import os

# pick first class
first_class = os.listdir(data_dir)[0]
class_path = os.path.join(data_dir, first_class)

# pick first image inside that class
first_image = os.listdir(class_path)[0]
image_path = os.path.join(class_path, first_image)

print("Testing:", image_path)
print("Prediction:", predict_image(image_path))


Testing: /kaggle/input/rice-disease-dataset/Rice_Leaf_AUG/Leaf scald/aug_0_5995.jpg
Prediction: Leaf scald
